In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / "python"))

from db_connection import get_database_engine

engine = get_database_engine()


Task 1 

In [ ]:
df_track = pd.read_sql("select * from dwh.dim_track", engine)
top_artists_albums = df_track.groupby("artist_name")["album_id"]\
    .nunique().reset_index(name="albums_count").sort_values("albums_count", ascending=False).head(5)
top_artists_albums

In [ ]:
top_artists_tracks = df_track.groupby("artist_name")["track_id"]\
    .count().reset_index(name="track_count").sort_values("track_count", ascending=False).head(5)
top_artists_tracks

In [ ]:
top_genres_tracks = df_track.groupby("genre_name")["track_id"]\
    .count().reset_index(name="track_count").sort_values("track_count", ascending=False).head(5)
top_genres_tracks

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.barplot(x="albums_count", y="artist_name", data=top_artists_albums, ax=axes[0])
axes[0].set_title("Top 5 Artists by Number of Albums")
sns.barplot(x="track_count", y="artist_name", data=top_artists_tracks, ax=axes[1])
axes[1].set_title("Top 5 Artists by Number of Tracks")
sns.barplot(x="track_count", y="genre_name", data=top_genres_tracks, ax=axes[2])
axes[2].set_title("Top 5 Genres by Number of Tracks")
axes[0].set_xlabel("Albums Count")
axes[1].set_xlabel("Tracks Count")
axes[2].set_xlabel("Tracks Count")
axes[1].set_ylabel("")
axes[2].set_ylabel("")
plt.tight_layout()
plt.show()

Task 2

In [ ]:
df_invoice = pd.read_sql(
    "SELECT * FROM dwh.fact_invoice",
    engine
)

In [ ]:
top_amount_customers = df_invoice.groupby("customer_id")[["total_usd","total_ils"]].sum()\
    .reset_index().sort_values("total_usd", ascending=False).head(5)
top_amount_customers

In [ ]:
df_customer = pd.read_sql(
    "select * from dwh.dim_customer",
    engine
)
top_amount_customers = top_amount_customers.merge(df_customer[["customer_id", "first_name", "last_name"]], on="customer_id")
top_amount_customers["full_name"] = top_amount_customers["first_name"] + " " + top_amount_customers["last_name"]
top_amount_customers = top_amount_customers[["full_name", "total_usd", "total_ils"]]
top_amount_customers

In [ ]:
top_amount_customers.plot(
    kind="barh",
    x="full_name",
    y=["total_usd", "total_ils"]
)
plt.title("Top 5 Customers by Total Amount Spent")
plt.xlabel("Total Amount")
plt.ylabel("Customer")
plt.legend(["Total USD", "Total ILS"])
plt.tight_layout()
plt.show()

Task 3

In [ ]:
df_invoice["invoice_date"] = pd.to_datetime(df_invoice["invoice_date"])
df_invoice["month"] = df_invoice["invoice_date"].dt.month
df_invoice["year"] = df_invoice["invoice_date"].dt.year
monthly_revenue_usd = df_invoice.groupby(["month", "year"])["total_usd"].sum().reset_index()
monthly_revenue_usd_pivot = monthly_revenue_usd.pivot(index="month", columns="year", values="total_usd")
monthly_revenue_ils = df_invoice.groupby(["month", "year"])["total_ils"].sum().reset_index()
monthly_revenue_ils_pivot = monthly_revenue_ils.pivot(index="month", columns="year", values="total_ils")
fig, axes = plt.subplots(1, 2, figsize=(14,5))
monthly_revenue_usd_pivot.plot(ax=axes[0])
monthly_revenue_ils_pivot.plot(ax=axes[1])
axes[0].set_title("Monthly Revenue in USD")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Revenue (USD)")
axes[1].set_title("Monthly Revenue in ILS")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Revenue (ILS)")
plt.tight_layout()
plt.show()

Task 4

In [ ]:
df_invoice_line = pd.read_sql(
    "SELECT * FROM dwh.fact_invoice_line",
    engine
)
df_track_sales = df_track.merge(df_invoice_line[[ "track_id","total_line"]], on="track_id", how="left")
df_track_sales["total_line"] = df_track_sales["total_line"].fillna(0)
top_selling_tracks = df_track_sales.groupby(["track_name","duration_seconds"])["total_line"]\
    .sum().reset_index(name="sales_amount").sort_values("sales_amount", ascending=False)

correlation = top_selling_tracks["duration_seconds"].corr(
    top_selling_tracks["sales_amount"]
)
print(f"Correlation between track duration and sales amount: {correlation:.2f}")
sns.scatterplot(
    data=top_selling_tracks,
    x="duration_seconds",
    y="sales_amount"
)

plt.title("Correlation Between Track Duration and Sales Amount")
plt.xlabel("Duration Seconds")
plt.ylabel("Sales Amount USD")
plt.tight_layout()
plt.show()

A weak positive correlation (0.15) was found between track duration and sales amount, indicating that longer tracks tend to generate slightly higher sales, but the relationship is not strong.

Task 5

In [ ]:
df_purchase_history = df_customer.merge(df_invoice, on="customer_id", how="left")\
    .merge(df_invoice_line, on="invoice_id", how="left")\
    .merge(df_track[["track_id","track_name", "artist_name", "genre_name"]], on="track_id", how="left")

df_favorite_genres = df_purchase_history.groupby(["customer_id","genre_name"])["total_usd"]\
    .sum().reset_index(name="total_spent").sort_values("total_spent", ascending=False)

df_top_customer_genres = df_favorite_genres.sort_values(["customer_id", "total_spent"], ascending=[True, False])\
    .groupby("customer_id").head(2)

popular_tracks_by_genre = df_purchase_history.groupby(["genre_name","track_id","track_name"])["total_usd"]\
    .sum().reset_index(name="total_spent").sort_values(["genre_name", "total_spent"], ascending=[True, False])

df_top_customer_tracks_by_genre = df_top_customer_genres.merge(popular_tracks_by_genre, on="genre_name", how="left")

purchased_tracks = df_purchase_history[["customer_id", "track_id"]].drop_duplicates()

recommendation_tracks = df_top_customer_tracks_by_genre.\
    merge(purchased_tracks, on=["customer_id", "track_id"], how="left", indicator=True)

recommendation_tracks.columns
recommendation_tracks = recommendation_tracks.rename\
(columns={"total_spent_x": "customer_genre_spent","total_spent_y": "track_genre_sales"})

recommendation_tracks = recommendation_tracks[recommendation_tracks["_merge"] == "left_only"].drop(columns=["_merge"])
recommendation_tracks = recommendation_tracks.\
    sort_values(["customer_id", "genre_name", "track_genre_sales"],ascending=[True, True, False]
)

final_recommendations = recommendation_tracks.groupby(["customer_id","genre_name"]).head(3)
final_recommendations.groupby("customer_id")["track_id"].count().head()
final_recommendations.head(12)





Task 6
Segment customers by total spending

In [ ]:
customer_spending = df_invoice.groupby("customer_id")[["total_usd","total_ils"]].sum().reset_index()
customer_spending["total_usd"].describe()


In [ ]:
bins_amount = [0,10,20,30,float("inf")]
bins_labels = ["Low", "Medium", "High", "VIP"]

customer_spending["spending_category"] = pd.cut(customer_spending["total_usd"], bins=bins_amount, labels=bins_labels, include_lowest=True)
customer_spending.head()

In [ ]:
customers_in_segments = customer_spending.groupby("spending_category")["customer_id"].count().reset_index(name="customer_count")
total_revenue_by_segment = customer_spending.groupby("spending_category")["total_usd"].sum().reset_index(name="total_usd")
segment_analysis = customers_in_segments.merge(total_revenue_by_segment, on="spending_category")
segment_analysis["average_spent_per_customer"] = segment_analysis["total_usd"] / segment_analysis["customer_count"]
segment_analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.barplot(x="spending_category", y="customer_count", data=segment_analysis, ax=axes[0])
axes[0].set_title("Customer Count by Spending Category")
axes[0].set_xlabel("Spending Category")
axes[0].set_ylabel("Customer Count")
sns.barplot(x="spending_category", y="total_usd", data=segment_analysis, ax=axes[1])
axes[1].set_title("Total Revenue by Segment")
axes[1].set_xlabel("Spending Category")
axes[1].set_ylabel("Total Revenue (USD)")
plt.tight_layout()
plt.show()

VIP customers represent a smaller share of customers but generate the highest total revenue, indicating their high business value.